In [12]:
import gc
import re
import torch
import pandas as pd

from transformers import (
    MarianMTModel,
    MarianTokenizer
)

from tqdm import tqdm


C:\Users\Gregorius Christian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
df = pd.read_csv("career_final_dataset.csv")

In [14]:
df.head()

,title,body,dominant_topic_name,url,good_score,bad_score,keyword_match,negative_match,final_score,career_text,predicted_role,role_score,top3_roles,top3_scores,text,problem_category,current_level,blocker_type
0,what are the skills required for vfx artist?,passionate about vfx. wanted to know more abou...,Career Change for Non-Corporate Artist,https://www.reddit.com/r/careerguidance/commen...,0.3588,0.1249,False,False,0.3307,what are the skills required for vfx artist? p...,vfx artist,0.7777,"['vfx artist', 'visual effects animator', 'gra...","[0.7777, 0.6448, 0.6029]",what are the skills required for vfx artist? p...,direction_confused,intermediate,no_portfolio
1,web developers out there can you provide guida...,i am a 4th year b.tech student (electronics an...,Software Development Career Transition,https://www.reddit.com/r/careerguidance/commen...,0.5111,0.2009,True,False,0.5326,web developers out there can you provide guida...,entry level web developer,0.7579,"['entry level web developer', 'web developer',...","[0.7579, 0.7473, 0.7355]",web developers out there can you provide guida...,beginner_lost,zero,no_foundation
2,interested in developing a career in the busin...,hello! i m a business administration and i m r...,Career Advancement in Data Analytics,https://www.reddit.com/r/careerguidance/commen...,0.5177,0.3283,True,False,0.4385,interested in developing a career in the busin...,business intelligence analyst,0.7509,"['business intelligence analyst', 'power bi an...","[0.7509, 0.6451, 0.589]",interested in developing a career in the busin...,direction_confused,intermediate,no_foundation
3,what are possible career s paths for data busi...,hi i am working in a company as data analyst d...,Career Advancement in Data Analytics,https://www.reddit.com/r/careerguidance/commen...,0.6199,0.3609,True,False,0.5352,what are possible career s paths for data busi...,business intelligence analyst,0.7503,"['business intelligence analyst', 'power bi an...","[0.7503, 0.6684, 0.6515]",what are possible career s paths for data busi...,direction_confused,intermediate,no_portfolio
4,what are the best opportunities for a manageme...,hi r careerguidance! i am currently a manageme...,Career Advancement in Data Analytics,http://www.reddit.com/r/careerguidance/comment...,0.4888,0.3160,True,False,0.4138,what are the best opportunities for a manageme...,business intelligence analyst,0.7425,"['business intelligence analyst', 'power bi an...","[0.7425, 0.6556, 0.5415]",what are the best opportunities for a manageme...,direction_confused,intermediate,no_portfolio


In [15]:
df['role'] = df['predicted_role']
df['pretext'] = df['text']

In [16]:
df = df[['pretext', 'role', 'problem_category', 'current_level', 'blocker_type']]

In [17]:
df.head(20)

,pretext,role,problem_category,current_level,blocker_type
0,what are the skills required for vfx artist? p...,vfx artist,direction_confused,intermediate,no_portfolio
1,web developers out there can you provide guida...,entry level web developer,beginner_lost,zero,no_foundation
2,interested in developing a career in the busin...,business intelligence analyst,direction_confused,intermediate,no_foundation
3,what are possible career s paths for data busi...,business intelligence analyst,direction_confused,intermediate,no_portfolio
4,what are the best opportunities for a manageme...,business intelligence analyst,direction_confused,intermediate,no_portfolio
5,what should i learn for mobile app development...,mobile developer,direction_confused,intermediate,no_foundation
6,what are possible career paths for a data busi...,business intelligence analyst,direction_confused,intermediate,no_portfolio
7,what is the demand for powerbi? what jobs dema...,power bi analyst,direction_confused,intermediate,no_portfolio
8,please help me out to build my career as a web...,entry level web developer,beginner_lost,intermediate,no_foundation
9,data analysts what are the most important exce...,data analyst,overwhelmed,intermediate,no_foundation


In [18]:
df.shape

(2942, 5)

In [19]:
# LOAD MODEL
model_name = "Helsinki-NLP/opus-mt-en-id"

tokenizer = MarianTokenizer.from_pretrained(
    model_name
)

model = MarianMTModel.from_pretrained(
    model_name
)

model.eval()

# SPECIAL TOKENS
special_tokens = [

    "[ML_ENGINEER]",
    "[ML]",
    "[DL]",
    "[AI]",
    "[SE]",
    "[BE_DEV]",
    "[FE_DEV]",
    "[FS_DEV]",
    "[DS]",
    "[DA]",
    "[DE]",
    "[CE]",
    "[DEVOPS]",
    "[NE]",
    "[UIUX]",
    "[CYBER]",
    "[CLOUD]",
    "[API]",
    "[SQL]",
    "[DOCKER]",
    "[K8S]",
    "[AWS]"

]

tokenizer.add_special_tokens({
    "additional_special_tokens": special_tokens
})

model.resize_token_embeddings(
    len(tokenizer)
)

# PROTECTED TERMS
protected_terms = {

    "machine learning engineer":
        "[ML_ENGINEER]",

    "machine learning":
        "[ML]",

    "deep learning":
        "[DL]",

    "artificial intelligence":
        "[AI]",

    "software engineer":
        "[SE]",

    "backend developer":
        "[BE_DEV]",

    "frontend developer":
        "[FE_DEV]",

    "fullstack developer":
        "[FS_DEV]",

    "data scientist":
        "[DS]",

    "data analyst":
        "[DA]",

    "data engineer":
        "[DE]",

    "cloud engineer":
        "[CE]",

    "devops engineer":
        "[DEVOPS]",

    "network engineer":
        "[NE]",

    "ui ux designer":
        "[UIUX]",

    "cybersecurity":
        "[CYBER]",

    "cloud computing":
        "[CLOUD]",

    "api":
        "[API]",

    "sql":
        "[SQL]",

    "docker":
        "[DOCKER]",

    "kubernetes":
        "[K8S]",

    "aws":
        "[AWS]",

}

# RESTORE MAP
restore_terms_map = {

    "[ML_ENGINEER]":
        "Machine Learning Engineer",

    "[ML]":
        "Machine Learning",

    "[DL]":
        "Deep Learning",

    "[AI]":
        "Artificial Intelligence",

    "[SE]":
        "Software Engineer",

    "[BE_DEV]":
        "Backend Developer",

    "[FE_DEV]":
        "Frontend Developer",

    "[FS_DEV]":
        "Fullstack Developer",

    "[DS]":
        "Data Scientist",

    "[DA]":
        "Data Analyst",

    "[DE]":
        "Data Engineer",

    "[CE]":
        "Cloud Engineer",

    "[DEVOPS]":
        "DevOps Engineer",

    "[NE]":
        "Network Engineer",

    "[UIUX]":
        "UI UX Designer",

    "[CYBER]":
        "Cybersecurity",

    "[CLOUD]":
        "Cloud Computing",

    "[API]":
        "API",

    "[SQL]":
        "SQL",

    "[DOCKER]":
        "Docker",

    "[K8S]":
        "Kubernetes",

    "[AWS]":
        "AWS",

}

# REPAIR MAP
repair_map = {

    "insinyur pembelajaran mesin":
        "Machine Learning Engineer",

    "pembelajaran mesin":
        "Machine Learning",

    "pembelajaran mendalam":
        "Deep Learning",

    "kecerdasan buatan":
        "Artificial Intelligence",

    "pengembang backend":
        "Backend Developer",

    "pengembang frontend":
        "Frontend Developer",

    "pengembang fullstack":
        "Fullstack Developer",

    "ilmuwan data":
        "Data Scientist",

    "analis data":
        "Data Analyst",

    "insinyur perangkat lunak":
        "Software Engineer",

}

# PROTECT TERMS
def protect_terms(text):

    text = str(text)

    sorted_terms = sorted(
        protected_terms.items(),
        key=lambda x: len(x[0]),
        reverse=True
    )

    for term, token in sorted_terms:

        text = re.sub(
            re.escape(term),
            token,
            text,
            flags=re.IGNORECASE
        )

    return text

# RESTORE TERMS
def restore_terms(text):

    text = str(text)

    for token, term in restore_terms_map.items():

        text = text.replace(
            token,
            term
        )

    return text

# REPAIR TECH TERMS
def repair_technical_terms(text):

    text = str(text)

    for indo, eng in repair_map.items():

        text = re.sub(
            indo,
            eng,
            text,
            flags=re.IGNORECASE
        )

    return text

# CLEAN TEXT
def clean_translation(text):

    text = str(text)

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    text = re.sub(
        r"\s+([?.!,])",
        r"\1",
        text
    )

    text = re.sub(r"\bi\s+m\b", "i am", text)
    text = re.sub(r"\bim\b", "i am", text)
    text = re.sub(r"\bi'm\b", "i am", text)

    return text.strip()

# TRANSLATE QUESTIONS
# TANPA PROTECT TOKEN
# def translate_pretext_batch(texts):

#     inputs = tokenizer(
#         texts,
#         return_tensors="pt",
#         padding=True,
#         truncation=True,
#         max_length=128
#     )

#     with torch.no_grad():

#         translated = model.generate(
#             **inputs,
#             max_new_tokens=128,
#             num_beams=4
#         )

#     outputs = [
#         tokenizer.decode(
#             t,
#             skip_special_tokens=True
#         )
#         for t in translated
#     ]

#     outputs = [
#         repair_technical_terms(o)
#         for o in outputs
#     ]

#     outputs = [
#         clean_translation(o)
#         for o in outputs
#     ]

#     return outputs

# TRANSLATE ANSWERS
# DENGAN PROTECT TOKEN
def translate_pretext_batch(texts):

    texts = [
        protect_terms(t)
        for t in texts
    ]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    with torch.no_grad():

        translated = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    outputs = [
        tokenizer.decode(
            t,
            skip_special_tokens=True
        )
        for t in translated
    ]

    outputs = [
        restore_terms(o)
        for o in outputs
    ]

    outputs = [
        repair_technical_terms(o)
        for o in outputs
    ]

    outputs = [
        clean_translation(o)
        for o in outputs
    ]

    return outputs

# TRANSLATE QUESTIONS
batch_size = 4

translated_pretext = []

pretext = df["pretext"].tolist()

print("Translating questions...")

for i in tqdm(
    range(0, len(pretext), batch_size)
):

    batch = pretext[i:i+batch_size]

    translated = translate_pretext_batch(batch)

    translated_pretext.extend(translated)

    gc.collect()

# SAVE RESULT
df["pretext_id"] = translated_pretext

# FINAL CLEAN
df["pretext_id"] = (
    df["pretext_id"]
    .astype(str)
    .str.strip()
)

# SAVE CSV
df.to_csv(
    "csv/career_it_indonesia_final2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nDONE!")
print("Saved to: csv/career_it_indonesia_final.csv")

C:\Users\Gregorius Christian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Translating questions...


100%|██████████| 736/736 [2:17:27<00:00, 11.21s/it]  



DONE!
Saved to: csv/career_it_indonesia_final.csv


In [20]:
import pandas as pd


In [21]:
df_test = pd.read_csv("csv/career_it_indonesia_final.csv")

In [22]:
df_test.head()

,pretext,role,problem_category,current_level,blocker_type,pretext_id
0,what are the skills required for vfx artist? p...,vfx artist,direction_confused,intermediate,no_portfolio,apa dibutuhkan keterampilan untuk seniman vfx?...
1,web developers out there can you provide guida...,entry level web developer,beginner_lost,zero,no_foundation,Pengembang web di luar sana dapat Anda memberi...
2,interested in developing a career in the busin...,business intelligence analyst,direction_confused,intermediate,no_foundation,Tertarik dalam mengembangkan karir dalam kecer...
3,what are possible career s paths for data busi...,business intelligence analyst,direction_confused,intermediate,no_portfolio,apa kemungkinan jalur karir untuk analis bisni...
4,what are the best opportunities for a manageme...,business intelligence analyst,direction_confused,intermediate,no_portfolio,apa peluang terbaik untuk manajer analis denga...


In [23]:
df_test = df_test.drop('pretext', axis=1)

In [24]:
df_test['pretext'] = df_test['pretext_id']

In [25]:
df_test.head()

,role,problem_category,current_level,blocker_type,pretext_id,pretext
0,vfx artist,direction_confused,intermediate,no_portfolio,apa dibutuhkan keterampilan untuk seniman vfx?...,apa dibutuhkan keterampilan untuk seniman vfx?...
1,entry level web developer,beginner_lost,zero,no_foundation,Pengembang web di luar sana dapat Anda memberi...,Pengembang web di luar sana dapat Anda memberi...
2,business intelligence analyst,direction_confused,intermediate,no_foundation,Tertarik dalam mengembangkan karir dalam kecer...,Tertarik dalam mengembangkan karir dalam kecer...
3,business intelligence analyst,direction_confused,intermediate,no_portfolio,apa kemungkinan jalur karir untuk analis bisni...,apa kemungkinan jalur karir untuk analis bisni...
4,business intelligence analyst,direction_confused,intermediate,no_portfolio,apa peluang terbaik untuk manajer analis denga...,apa peluang terbaik untuk manajer analis denga...


In [26]:
df_test = df_test.drop('pretext_id', axis=1)

In [27]:
df_test = df_test[['pretext', 'role', 'problem_category', 'current_level', 'blocker_type']]

In [28]:
df_test.head()

,pretext,role,problem_category,current_level,blocker_type
0,apa dibutuhkan keterampilan untuk seniman vfx?...,vfx artist,direction_confused,intermediate,no_portfolio
1,Pengembang web di luar sana dapat Anda memberi...,entry level web developer,beginner_lost,zero,no_foundation
2,Tertarik dalam mengembangkan karir dalam kecer...,business intelligence analyst,direction_confused,intermediate,no_foundation
3,apa kemungkinan jalur karir untuk analis bisni...,business intelligence analyst,direction_confused,intermediate,no_portfolio
4,apa peluang terbaik untuk manajer analis denga...,business intelligence analyst,direction_confused,intermediate,no_portfolio


In [29]:
df_test.to_csv("csv/dataset_pretext.csv")

In [46]:
import pandas as pd

df_testing = pd.read_csv("csv/career_it_indonesia_final2.csv")

df_testing.head()

,pretext,role,problem_category,current_level,blocker_type,pretext_id
0,what are the skills required for vfx artist? p...,vfx artist,direction_confused,intermediate,no_portfolio,apa dibutuhkan keterampilan untuk seniman vfx?...
1,web developers out there can you provide guida...,entry level web developer,beginner_lost,zero,no_foundation,Pengembang web di luar sana dapat Anda memberi...
2,interested in developing a career in the busin...,business intelligence analyst,direction_confused,intermediate,no_foundation,Tertarik dalam mengembangkan karir dalam kecer...
3,what are possible career s paths for data busi...,business intelligence analyst,direction_confused,intermediate,no_portfolio,apa kemungkinan jalur karir untuk analis bisni...
4,what are the best opportunities for a manageme...,business intelligence analyst,direction_confused,intermediate,no_portfolio,apa peluang terbaik untuk manajer analis denga...


In [47]:
df_testing = df_testing.drop('pretext', axis=1)
df_testing['pretext'] = df_testing['pretext_id']
df_testing['target_role'] = df_testing['role']
df_testing = df_testing[['pretext', 'target_role', 'problem_category', 'current_level', 'blocker_type']]

In [48]:
df_testing.head(20)

,pretext,target_role,problem_category,current_level,blocker_type
0,apa dibutuhkan keterampilan untuk seniman vfx?...,vfx artist,direction_confused,intermediate,no_portfolio
1,Pengembang web di luar sana dapat Anda memberi...,entry level web developer,beginner_lost,zero,no_foundation
2,Tertarik dalam mengembangkan karir dalam kecer...,business intelligence analyst,direction_confused,intermediate,no_foundation
3,apa kemungkinan jalur karir untuk analis bisni...,business intelligence analyst,direction_confused,intermediate,no_portfolio
4,apa peluang terbaik untuk manajer analis denga...,business intelligence analyst,direction_confused,intermediate,no_portfolio
5,apa yang harus saya pelajari untuk pengembanga...,mobile developer,direction_confused,intermediate,no_foundation
6,apa yang mungkin jalur karir untuk seorang ana...,business intelligence analyst,direction_confused,intermediate,no_portfolio
7,Apa permintaan akan powerbi? Pekerjaan apa yan...,power bi analyst,direction_confused,intermediate,no_portfolio
8,Tolong bantu saya membangun karir saya sebagai...,entry level web developer,beginner_lost,intermediate,no_foundation
9,Apa yang paling penting dari konsep perangkat ...,data analyst,overwhelmed,intermediate,no_foundation


In [50]:
# jumlah data unknown
unknown_count = (
    df_testing['target_role'] == 'unknown'
).sum()

print("Jumlah unknown:", unknown_count)

Jumlah unknown: 880


In [54]:
# hapus data unknown
df_testing = df_testing[
    df_testing['target_role'] != 'unknown'
]

# reset index
df_testing = df_testing.reset_index(drop=True)

print("Total data setelah drop:", len(df_testing))

Total data setelah drop: 2062


In [56]:
df_testing['target_role'].value_counts

<bound method IndexOpsMixin.value_counts of 0                             vfx artist
1              entry level web developer
2          business intelligence analyst
3          business intelligence analyst
4          business intelligence analyst
                      ...               
2057                    ux/ui specialist
2058                       it technician
2059    artificial intelligence engineer
2060                multimedia architect
2061                      game developer
Name: target_role, Length: 2062, dtype: object>

In [57]:
df_testing['current_level'].value_counts

<bound method IndexOpsMixin.value_counts of 0       intermediate
1               zero
2       intermediate
3       intermediate
4       intermediate
            ...     
2057    intermediate
2058    intermediate
2059    intermediate
2060    intermediate
2061    intermediate
Name: current_level, Length: 2062, dtype: object>

In [58]:
df_testing['blocker_type'].value_counts

<bound method IndexOpsMixin.value_counts of 0        no_portfolio
1       no_foundation
2       no_foundation
3        no_portfolio
4        no_portfolio
            ...      
2057    no_foundation
2058     no_portfolio
2059     no_portfolio
2060     no_portfolio
2061     no_portfolio
Name: blocker_type, Length: 2062, dtype: object>

In [59]:
df_testing.to_csv('csv/dataset_pretext.csv')